# Plot Influence Run Discovery

This notebook starts by discovering the run directories we care about.

- ES runs: trained on `countdown` or `proofwriter`, `population_size == 30`, `mirror_sampling == False`, and no weight decay.
- GRPO runs: any run under `grpo_experiments/runs` trained on `countdown` or `proofwriter`.

The later cells display the matching paths and build a per-run influence dataframe with one row per prior task.


In [11]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display


TARGET_TASKS = ("countdown", "proofwriter")
ALL_TASKS = (
    "countdown",
    "gsm8k",
    "proofwriter",
    "hellaswag",
    "piqa",
    "arc-challenge",
    "mmlu-pro",
)
GLOBAL_STEP_RE = re.compile(r"_global_step_(\d+)$")
SEED_RE = re.compile(r"seed(\d+)$")
SIZE_RE = re.compile(r"(1\.5B|3B|7B)")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "evaluation").is_dir() and (candidate / "experiments").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the es_finetuning repo.")


REPO_ROOT = find_repo_root()
ES_ROOT = REPO_ROOT / "experiments" / "test"
GRPO_ROOT = REPO_ROOT / "grpo_experiments" / "runs"


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


def load_jsonl_records(path: Path) -> list[dict]:
    records = []
    for line in path.read_text().splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def read_text(path: Path) -> str:
    return path.read_text().strip()


def parse_seed(name: str) -> int | None:
    for token in name.split("_"):
        match = SEED_RE.fullmatch(token)
        if match is not None:
            return int(match.group(1))
    return None


def relative_to_repo(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def iter_directories(root: Path):
    if not root.exists():
        return
    for path in sorted(root.iterdir()):
        if path.is_dir():
            yield path


def is_es_run_dir(path: Path) -> bool:
    return (path / "args.json").is_file() and (path / "iteration_updates").is_dir()


def has_grpo_checkpoints(path: Path) -> bool:
    for child in sorted(path.iterdir()):
        if child.is_dir() and GLOBAL_STEP_RE.search(child.name) and (child / "config.json").is_file():
            return True
    return False


def is_grpo_run_dir(path: Path) -> bool:
    return (path / "model.txt").is_file() and has_grpo_checkpoints(path)


def uses_weight_decay(args: dict) -> bool:
    weight_decay_type = str(args.get("weight_decay_type", "none")).lower()
    weight_decay_lambda = float(args.get("weight_decay_lambda", 0.0) or 0.0)
    return weight_decay_type != "none" or abs(weight_decay_lambda) > 0.0


def detect_task(identifier: str, tasks: tuple[str, ...] = TARGET_TASKS) -> str | None:
    normalized = identifier.replace("_", "-").lower()
    matches = [task for task in tasks if task in normalized]
    if len(matches) == 1:
        return matches[0]
    return None


def infer_family(model_name: str) -> str:
    normalized = model_name.lower()
    if "qwen" in normalized:
        return "qwen"
    if "llama" in normalized:
        return "llama"
    raise ValueError(f"Could not infer family from model name: {model_name}")


def infer_size(model_name: str) -> str:
    match = SIZE_RE.search(model_name)
    if match is None:
        raise ValueError(f"Could not infer size from model name: {model_name}")
    return match.group(1)


def missing_metric_tasks(run_dir: Path, tasks: tuple[str, ...] = ALL_TASKS) -> list[str]:
    return [task for task in tasks if not (run_dir / f"{task}.jsonl").is_file()]


def load_accuracy_delta(run_dir: Path, task: str) -> float:
    path = run_dir / f"{task}.jsonl"
    records = load_jsonl_records(path)
    if not records:
        raise ValueError(f"Empty metrics file for {relative_to_repo(run_dir)}: {path.name}")

    frame = pd.DataFrame(records)
    if "accuracy" not in frame.columns:
        raise ValueError(f"Missing accuracy column for {relative_to_repo(run_dir)}: {path.name}")

    original_accuracy = float(frame.iloc[0]["accuracy"])
    final_accuracy = float(frame.iloc[-1]["accuracy"])
    return final_accuracy - original_accuracy


def find_es_run_paths(
    tasks: tuple[str, ...] = TARGET_TASKS,
    population_size: int = 30,
    mirror_sampling: bool = False,
    require_no_weight_decay: bool = True,
    root: Path = ES_ROOT,
) -> list[Path]:
    matches: list[Path] = []
    for run_dir in iter_directories(root):
        if not is_es_run_dir(run_dir):
            continue
        args = load_json(run_dir / "args.json")
        if args.get("task") not in tasks:
            continue
        if int(args.get("population_size", -1)) != population_size:
            continue
        if bool(args.get("mirror_sampling")) != mirror_sampling:
            continue
        if require_no_weight_decay and uses_weight_decay(args):
            continue
        matches.append(run_dir.resolve())
    return sorted(
        matches,
        key=lambda path: (
            load_json(path / "args.json")["task"],
            load_json(path / "args.json")["model_name"],
            parse_seed(path.name) or -1,
            path.name,
        ),
    )


def find_grpo_run_paths(
    tasks: tuple[str, ...] = TARGET_TASKS,
    root: Path = GRPO_ROOT,
) -> list[Path]:
    matches: list[Path] = []
    for run_dir in iter_directories(root):
        if not is_grpo_run_dir(run_dir):
            continue
        if detect_task(run_dir.name, tasks=tasks) is None:
            continue
        matches.append(run_dir.resolve())
    return sorted(matches, key=lambda path: (detect_task(path.name, tasks=tasks) or "", path.name))


def summarize_es_runs(run_paths: list[Path]) -> pd.DataFrame:
    records = []
    for run_dir in run_paths:
        args = load_json(run_dir / "args.json")
        records.append(
            {
                "task": args["task"],
                "model": args["model_name"],
                "seed": args.get("global_seed"),
                "population_size": args.get("population_size"),
                "mirror_sampling": args.get("mirror_sampling"),
                "weight_decay_type": args.get("weight_decay_type"),
                "weight_decay_lambda": args.get("weight_decay_lambda"),
                "path": relative_to_repo(run_dir),
            }
        )
    return pd.DataFrame(records)


def summarize_grpo_runs(run_paths: list[Path]) -> pd.DataFrame:
    records = []
    for run_dir in run_paths:
        records.append(
            {
                "task": detect_task(run_dir.name),
                "model": read_text(run_dir / "model.txt"),
                "seed": parse_seed(run_dir.name),
                "path": relative_to_repo(run_dir),
            }
        )
    return pd.DataFrame(records)


def describe_run(run_dir: Path, algorithm: str) -> dict[str, str]:
    if algorithm == "es":
        args = load_json(run_dir / "args.json")
        target = args["task"]
        model_name = args["model_name"]
    elif algorithm == "grpo":
        target = detect_task(run_dir.name, tasks=TARGET_TASKS)
        model_name = read_text(run_dir / "model.txt")
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")

    if target is None:
        raise ValueError(f"Could not infer target task for {relative_to_repo(run_dir)}")

    return {
        "algorithm": algorithm,
        "family": infer_family(model_name),
        "size": infer_size(model_name),
        "target": target,
    }


def build_influence_df(es_run_paths: list[Path], grpo_run_paths: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    records = []
    skipped_runs = []

    for algorithm, run_paths in (("es", es_run_paths), ("grpo", grpo_run_paths)):
        for run_dir in run_paths:
            missing_tasks = missing_metric_tasks(run_dir)
            if missing_tasks:
                skipped_runs.append(
                    {
                        "algorithm": algorithm,
                        "path": relative_to_repo(run_dir),
                        "missing_tasks": ", ".join(missing_tasks),
                    }
                )
                continue

            run_metadata = describe_run(run_dir, algorithm)
            for prior in ALL_TASKS:
                if prior == run_metadata["target"]:
                    continue
                records.append(
                    {
                        **run_metadata,
                        "prior": prior,
                        "delta": load_accuracy_delta(run_dir, prior),
                    }
                )

    influence_df = pd.DataFrame(
        records,
        columns=["algorithm", "family", "size", "target", "prior", "delta"],
    )
    skipped_runs_df = pd.DataFrame(
        skipped_runs,
        columns=["algorithm", "path", "missing_tasks"],
    )
    return influence_df, skipped_runs_df


In [ ]:
es_run_paths = find_es_run_paths()
grpo_run_paths = find_grpo_run_paths()

es_runs_df = summarize_es_runs(es_run_paths)
grpo_runs_df = summarize_grpo_runs(grpo_run_paths)

print(f"Found {len(es_run_paths)} ES runs.")
display(es_runs_df)

print(f"Found {len(grpo_run_paths)} GRPO runs.")
display(grpo_runs_df)

es_run_paths, grpo_run_paths


In [ ]:
influence_df, skipped_influence_runs_df = build_influence_df(es_run_paths, grpo_run_paths)

expected_priors_per_run = len(ALL_TASKS) - 1
complete_run_count = len(es_run_paths) + len(grpo_run_paths) - len(skipped_influence_runs_df)
expected_rows = complete_run_count * expected_priors_per_run
assert len(influence_df) == expected_rows

print(
    f"Built {len(influence_df)} influence rows "
    f"({expected_priors_per_run} priors per complete run across {complete_run_count} complete runs)."
)
display(influence_df)

if not skipped_influence_runs_df.empty:
    print(f"Skipped {len(skipped_influence_runs_df)} incomplete run(s).")
    display(skipped_influence_runs_df)

influence_df


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

algorithm_order = ["es", "grpo"]
algorithm_labels = {"es": "ES (30)", "grpo": "GRPO"}
palette = {"es": "#000000", "grpo": "#9e9e9e"}


def draw_grouped_boxplots(ax, plot_df, group_column, order, xlabel):
    centers = list(range(1, len(order) + 1))
    offsets = {"es": -0.18, "grpo": 0.18}
    width = 0.3

    for algorithm in algorithm_order:
        series = [
            plot_df.loc[
                (plot_df[group_column] == group_value) & (plot_df["algorithm"] == algorithm),
                "delta",
            ].to_list()
            for group_value in order
        ]
        positions = [center + offsets[algorithm] for center in centers] if xlabel != "Overall" else centers 
        boxplot = ax.boxplot(
            series,
            positions=positions,
            widths=width * 2 if xlabel == "Overall" else width,
            patch_artist=True,
            manage_ticks=False,
        )
        for box in boxplot["boxes"]:
            box.set(facecolor=palette[algorithm], edgecolor="black", alpha=0.95)
        for median in boxplot["medians"]:
            median.set(color="white", linewidth=1.5)
        for whisker in boxplot["whiskers"]:
            whisker.set(color="black", linewidth=1)
        for cap in boxplot["caps"]:
            cap.set(color="black", linewidth=1)
        for flier in boxplot["fliers"]:
            flier.set(marker="o", markersize=3, markerfacecolor=palette[algorithm], markeredgecolor="black")

    ax.axhline(0.0, color="0.4", linestyle="--", linewidth=1)
    ax.set_xticks(centers)
    ax.set_xticklabels(order)
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", which="both", color="0.88")


target_plot_df = influence_df.assign(
    target_label=influence_df["target"].map(
        {"countdown": "Countdown", "proofwriter": "ProofWriter"}
    )
)

family_plot_df = influence_df.loc[influence_df["size"] == "3B"].copy()
family_plot_df["family_label"] = family_plot_df["family"].map(
    {"qwen": "Qwen", "llama": "Llama"}
)

size_plot_df = influence_df.loc[influence_df["family"] == "qwen"].copy()
size_plot_df["size_label"] = size_plot_df["size"]

fig, axes = plt.subplots(1, 4, figsize=(10, 2.5), sharey=True, width_ratios=[1,2,2,2])

influence_df["algorithm_names"] = influence_df["algorithm"].map(algorithm_labels)

plot_specs = [
    (influence_df, "algorithm_names", ["ES (30)", "GRPO"], "Overall"),
    (target_plot_df, "target_label", ["Countdown", "ProofWriter"], "Target Task"),
    (family_plot_df, "family_label", ["Qwen", "Llama"], "Model Family"),
    (size_plot_df, "size_label", ["1.5B", "3B", "7B"], "Model Size"),
]

for ax, (plot_df, x_column, order, title) in zip(axes, plot_specs):
    draw_grouped_boxplots(ax, plot_df, x_column, order, title)
axes[0].set_ylabel("$\\Delta$ Prior Task Accuracy")

handles = [
    Patch(facecolor=palette[algorithm], edgecolor="black", label=algorithm_labels[algorithm])
    for algorithm in algorithm_order
]
axes[-1].legend(
    handles=handles,
    loc="best",
    ncol=1,
    handletextpad=0.4,
    borderpad=0.3,
)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig("influence_plots.pdf", bbox_inches="tight")
plt.show()
